In [ ]:
from dataclasses import dataclass

@dataclass
class WearInjection:
    joints: list[int]
    injection_duration: float
    wait_duration: float
    fault_value: int

@dataclass
class Run:
    name: str
    wear_injections: list[WearInjection]

In [ ]:
# Define all runs for which data should be collected
runs = []

for joint in range(6): # [0, 1, 2, 3, 4, 5]
    run = Run(
        name=f"train_data_{joint}", 
        wear_injections=[
            WearInjection(joints=[], injection_duration=0, wait_duration=180, fault_value=0),
            WearInjection(joints=[joint], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 1) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 2) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 3) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 4) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 5) % 6], injection_duration=60, wait_duration=180, fault_value=1),
        ]
    )
    runs.append(run)

for joint in range(6): # [0, 1, 2, 3, 4, 5]
    run = Run(
        name=f"train_data_{joint+6}", 
        wear_injections=[
            WearInjection(joints=[], injection_duration=0, wait_duration=180, fault_value=0),
            WearInjection(joints=[joint], injection_duration=60, wait_duration=300, fault_value=1),
            WearInjection(joints=[joint], injection_duration=60, wait_duration=300, fault_value=1),
        ]
    )
    runs.append(run)

# Test runs
for joint in range(6): # [0, 1, 2, 3, 4, 5]
    run = Run(
        name=f"test_data_{joint}", 
        wear_injections=[
            WearInjection(joints=[], injection_duration=0, wait_duration=180, fault_value=0),
            WearInjection(joints=[joint], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 1) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 2) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 3) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 4) % 6], injection_duration=60, wait_duration=180, fault_value=1),
            WearInjection(joints=[(joint + 5) % 6], injection_duration=60, wait_duration=180, fault_value=1),
        ]
    )
    runs.append(run)

print([[wear_injection.joints for wear_injection in run.wear_injections] for run in runs])

In [ ]:
# Code for connecting to RabbitMQ and injecting fault
from communication.typed_protocol_client import TypedRabbitMQClient
from communication.typed_protocol import LoadProgram, LoadTCPProgram, InjectWear, Play
from communication.rabbitmq import Rabbitmq
from pathlib import Path
import yaml
def inject_wear(joints: list[1 | 2 | 3 | 4 | 5 | 6], fault_value: int, duration: int):
    def load_config(path: Path) -> dict:
        with path.open() as f:
            return yaml.safe_load(f)

    connect_config = load_config(Path("../../../communication/connect.yml"))
    connect_config["ip"] = "127.0.0.1"

    typed_client = TypedRabbitMQClient(Rabbitmq(**connect_config))
    typed_client.client.connect_to_server()

    msg = InjectWear(duration=duration, fault_value=fault_value, joints=joints)
    typed_client.publish(msg)

In [ ]:
import yaml
import time

yaml_path = "../../../DTsolution/DTservices/data_recorder/influxdb.yml"

# Read standard data_recorder yaml config
with open(yaml_path, "r") as f:   
    data_recorder_config = yaml.load(f, Loader=yaml.SafeLoader)

password = input()

# Run all the runs
for run in runs:
    # Make data_recorder write to the correct bucket
    data_recorder_config["bucket"] = run.name
    with open(yaml_path, 'w') as outfile:
        yaml.dump(data_recorder_config, outfile, default_flow_style=False)

    # Start run
    !echo {password} | sudo -S docker compose up --detach

    time.sleep(20) # Wait 20 seconds for everything to start before doing anything.

    # Inject wear faults periodically
    for wear_injection in run.wear_injections:
        inject_wear(joints=wear_injection.joints, fault_value=wear_injection.fault_value, duration=wear_injection.injection_duration)
        print(f"Wore joint: {wear_injection.joints}, with fault_value={wear_injection.fault_value}.")
        time.sleep(wear_injection.wait_duration) # Wait while moves happen

    # Stop the run
    !docker compose down

    time.sleep(30) # Wait 30 seconds for everything to stop before continuing

    

In [ ]:
!docker compose up --detach